[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/03_alumno_estructuras.ipynb)

# MLY1101 · Machine Learning — Actividad 1.2
## Estructuras de Datos y Almacenamiento en Python

**Resultado de aprendizaje (RA1):** recopila, a través de un trabajo colaborativo, sets de
datos representativos y de calidad, a partir de distintas fuentes, para responder a las
necesidades del contexto de negocio, considerando aspectos éticos.

**Indicador de logro (IL 1.2):** utiliza estructuras de datos en Python para el almacenamiento
y manipulación eficiente de datasets.

---

### La idea central de hoy

Elegir una estructura de datos **no es una decisión de estilo**. Decide cuánta RAM consume el
proceso, cuánto tarda en responder y —lo más traicionero— si el resultado es correcto.

Hoy vas a comprobar cuatro cosas, midiéndolas:

1. Una lista de Python ocupa **cuatro veces** lo que el mismo dato en un arreglo de NumPy.
2. Un ciclo `for` tarda **decenas de veces** más que la misma operación vectorizada.
3. Ajustar los tipos de columna reduce la memoria del dataset **a la mitad**.
4. Confundir `.loc` con `.iloc` **no da error**: da un resultado equivocado en silencio.

El cuarto punto es el que le cuesta el fin de semana a alguien todos los años.

---

### El caso

Mismo dataset de detecciones LiDAR de las actividades 1.1 y 1.3. Ahora la pregunta no es *de
dónde salen* ni *qué tan sucios están*, sino:

> **¿Estamos manipulando estos datos de una forma que aguante cuando en vez de 40.000
> detecciones sean 40 millones?**

---

### Al final de la sesión debes entregar

- El notebook con los 15 TODO resueltos y sus autochequeos en verde.
- Una **tabla de decisiones de almacenamiento** (última celda) con el formato que elegiste para
  tu proyecto y por qué, respaldado con las cifras que mediste tú.

---
## Preparación del entorno

Ejecuta esta celda primero. Funciona tanto en Google Colab como en Jupyter local.

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO
else:
    # El notebook vive en notebooks/, así que la raíz del repositorio es la carpeta superior.
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"

# Carpeta de trabajo para los archivos que vamos a exportar en el bloque 6.
SALIDAS = Path("salidas_act12")
SALIDAS.mkdir(exist_ok=True)

print("Colab:", EN_COLAB)
print("Raíz del repositorio:", RAIZ)
print("¿Existe el dataset?:", RUTA_DATOS.exists())

In [ ]:
import sys
import time

import numpy as np
import pandas as pd

import formatos  # comparación de formatos de almacenamiento: src/formatos.py

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

df = pd.read_csv(RUTA_DATOS)
print("pandas", pd.__version__, "| numpy", np.__version__)
print(f"Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")

---
# Bloque 1 · Por qué las listas de Python no sirven para esto

Una lista de Python es un **arreglo de punteros**: guarda direcciones de memoria que apuntan a
objetos dispersos por el montón. Cada flotante de esa lista es un objeto completo de Python, con
su cabecera, su contador de referencias y su tipo.

Un `ndarray` de NumPy es un **bloque contiguo** de bytes, todos del mismo tipo. El procesador lo
recorre aprovechando la caché, y las operaciones las ejecuta código compilado en C.

| Estructura | Memoria | Velocidad | Tipos |
|---|---|---|---|
| Lista de Python | Dispersa (punteros + objetos) | Lenta (un salto por elemento) | Heterogéneos |
| Arreglo de NumPy | Bloque contiguo | Rápida (C, vectorizada) | Homogéneos y fijos |

No lo creas porque lo dice la tabla. Mídelo.

### ✏️ TODO 1 — Medir la memoria

Compara cuánto ocupa la columna `speed_mps` como lista de Python y como arreglo de NumPy.

*Pista: `sys.getsizeof(lista)` **solo mide el arreglo de punteros**, no los objetos apuntados. Por
eso usamos `formatos.memoria_lista_python()`, que suma ambos. Para el arreglo, `.nbytes`.*

In [ ]:
# TODO 1: ¿cuánta memoria ocupa cada estructura?
velocidades_lista = df["speed_mps"].dropna().tolist()
velocidades_array = np.array(velocidades_lista, dtype="float64")

bytes_lista = formatos.memoria_lista_python(velocidades_lista)
bytes_array = velocidades_array.____          # atributo que da los bytes del bloque

print(f"Elementos: {len(velocidades_lista):,}\n")
print(f"Lista de Python : {bytes_lista/1024**2:6.2f} MB")
print(f"Arreglo NumPy   : {bytes_array/1024**2:6.2f} MB")
print(f"Factor          : {bytes_lista/bytes_array:6.1f}×")
print()
print(f"Engaño frecuente -> sys.getsizeof(lista) dice solo "
      f"{sys.getsizeof(velocidades_lista)/1024**2:.2f} MB: "
      f"mide los punteros, no los números.")

### ✏️ TODO 2 — Medir el tiempo

Convierte las velocidades de m/s a km/h (multiplicar por 3,6) de las dos maneras y compara.

*Pista: la versión vectorizada no lleva ningún `for`.*

In [ ]:
# TODO 2: la misma conversión, de las dos maneras.
inicio = time.perf_counter()
kmh_loop = [____ for v in velocidades_lista]      # con ciclo
seg_loop = time.perf_counter() - inicio

inicio = time.perf_counter()
kmh_vectorizado = ____                            # sin ciclo, sobre el ndarray
seg_vectorizado = time.perf_counter() - inicio

print(f"Ciclo de Python : {seg_loop*1000:8.2f} ms")
print(f"Vectorizado     : {seg_vectorizado*1000:8.2f} ms")
print(f"Factor          : {seg_loop/seg_vectorizado:8.0f}×")

In [ ]:
# Autochequeo
np.testing.assert_allclose(kmh_loop, kmh_vectorizado, rtol=1e-12,
                           err_msg="las dos versiones deben dar el mismo resultado")
assert bytes_lista > 3 * bytes_array, "revisa: la lista debería ocupar bastante más"
assert seg_vectorizado < seg_loop, "revisa: la versión vectorizada debería ser más rápida"
print("✅ Mismo resultado, mucha menos memoria y mucho menos tiempo.")

---
# Bloque 2 · Anatomía de un arreglo

Antes de entrenar cualquier modelo hay que auditar la forma de la matriz de variables. Tres
atributos bastan:

| Atributo | Qué devuelve | Para qué sirve |
|---|---|---|
| `.shape` | Tupla con el tamaño de cada dimensión | Comprobar que `X` e `y` tienen las mismas filas |
| `.ndim` | Cuántos ejes tiene | Distinguir un vector de una matriz |
| `.dtype` | El tipo exacto de los elementos | Es lo que decide cuánta RAM consume |

In [ ]:
columnas_numericas = ["box_length", "box_width", "box_height", "num_lidar_points"]
X = df[columnas_numericas].to_numpy()

print("shape:", X.shape, "-> (filas, columnas)")
print("ndim :", X.ndim, "-> es una matriz 2D")
print("dtype:", X.dtype)
print(f"memoria: {X.nbytes/1024**2:.2f} MB")

### ✏️ TODO 3 — El precio del `dtype`

`float64` usa 8 bytes por número; `float32`, 4. La mitad de memoria. ¿Es gratis?

Convierte `X` a `float32`, mide el ahorro y **mide también el error** que se introduce.

*Pista: `.astype("float32")` y `np.abs(original - convertido).max()`.*

In [ ]:
# TODO 3: ¿cuánto se ahorra y cuánto se paga al bajar la precisión?
X32 = X.astype("____")

error_max = np.abs(X - X32.astype("float64")).____()
print(f"float64: {X.nbytes/1024**2:.2f} MB")
print(f"float32: {X32.nbytes/1024**2:.2f} MB   (ahorro {100*(1 - X32.nbytes/X.nbytes):.0f} %)")
print(f"Error máximo introducido: {error_max:.2e}")
print(f"Magnitud típica del dato: {np.abs(X).mean():.2f}")

### ✏️ TODO 4 — Optimizar el DataFrame completo

Lo mismo, pero sobre todo el dataset. Dos cambios:

- las columnas de texto con **pocas categorías distintas** pasan a `category`;
- las columnas continuas pasan a `float32`.

*Pista: `.astype("category")` y `.astype("float32")`. Mide con `memory_usage(deep=True)`.*

In [ ]:
# TODO 4: reduce la memoria del DataFrame sin perder ninguna fila.
CATEGORICAS = ["object_type", "weather", "time_of_day", "detection_difficulty", "sensor_version"]
CONTINUAS = ["box_center_x", "box_center_y", "box_center_z",
             "box_length", "box_width", "box_height", "speed_mps"]

df_optimizado = df.copy()
for columna in CATEGORICAS:
    df_optimizado[columna] = df_optimizado[columna].astype("____")
for columna in CONTINUAS:
    df_optimizado[columna] = df_optimizado[columna].astype("____")

mem_antes = df.memory_usage(____).sum() / 1024**2
mem_despues = df_optimizado.memory_usage(____).sum() / 1024**2

print(f"Antes   : {mem_antes:6.1f} MB")
print(f"Después : {mem_despues:6.1f} MB")
print(f"Ahorro  : {100*(1 - mem_despues/mem_antes):5.0f} %")

In [ ]:
# Autochequeo
assert len(df_optimizado) == len(df), "optimizar memoria no puede perder filas"
assert mem_despues < mem_antes * 0.7, "revisa: deberías haber ahorrado más de un 30 %"
assert str(df_optimizado["object_type"].dtype) == "category", (
    "revisa: object_type debería quedar como category"
)
print(f"✅ {mem_antes:.1f} MB → {mem_despues:.1f} MB con las mismas {len(df):,} filas.")

---
# Bloque 3 · Series y DataFrame: el índice lo cambia todo

Una **Series** es un arreglo unidimensional **con índice explícito**. Esa es toda la diferencia
con un ndarray, y es una diferencia enorme: el índice es una etiqueta, y pandas la usa para
alinear operaciones automáticamente.

Un **DataFrame** es un contenedor de Series que comparten el mismo índice. De ahí que admita
columnas de tipos distintos, como una tabla de base de datos.

In [ ]:
ventas = pd.Series([450, 600, 320], index=["Ene", "Feb", "Mar"])
print(ventas["Feb"], "<- acceso por etiqueta, no por posición\n")

# El índice del DataFrame que venimos usando es el automático: 0, 1, 2, ...
print("Índice de df:", df.index[:5].tolist(), "...")

### ✏️ TODO 5 — La alineación automática

Suma dos Series cuyos índices **no coinciden del todo** y observa qué hace pandas.

*Pista: no da error. Mira dónde aparecen los `NaN`.*

In [ ]:
# TODO 5: ¿qué pasa al sumar dos Series con índices distintos?
a = pd.Series([10, 20, 30], index=["x", "y", "z"])
b = pd.Series([1, 2, 3], index=["y", "z", "w"])

suma = ____
print(suma)
print()
print("NumPy, en cambio, suma por posición y no sabe de etiquetas:")
print(a.to_numpy() + b.to_numpy())

In [ ]:
# Autochequeo
assert suma.isna().sum() == 2, "revisa: ¿cuántas etiquetas aparecen en una sola de las dos Series?"
assert suma["y"] == 21, "revisa: en 'y' deberían sumarse 20 y 1"
print("✅ pandas alineó por etiqueta y puso NaN donde una de las dos no tenía valor.")
print("   NumPy habría sumado por posición y devuelto 11, 22, 33: otro resultado, sin avisar.")

---
# Bloque 4 · ⭐ `.loc` contra `.iloc`

| | Selecciona por | Ejemplo |
|---|---|---|
| `.loc` | **Etiqueta** del índice y nombre de columna | `df.loc[0, "speed_mps"]` |
| `.iloc` | **Posición** entera, de 0 a n−1 | `df.iloc[0, 10]` |

En un DataFrame recién leído, la etiqueta y la posición **coinciden**: el índice es 0, 1, 2, …
Por eso `.loc` y `.iloc` parecen intercambiables y mucha gente aprende mal.

Dejan de coincidir en cuanto filtras, ordenas o eliminas filas. Y ahí empieza el problema.

In [ ]:
ciclistas = df[df["object_type"] == "CYCLIST"]

print(f"Ciclistas: {len(ciclistas)}")
print("Índice del resultado:", ciclistas.index[:5].tolist(), "...")
print()
print("Fíjate: la primera fila del filtro NO tiene etiqueta 0.")

### ✏️ TODO 6 — Posición contra etiqueta

Obtén la **primera fila** de `ciclistas` de las dos maneras y comprueba si son la misma.

*Pista: por posición siempre es `0`. Por etiqueta hay que usar la etiqueta que realmente existe:
`ciclistas.index[0]`.*

In [ ]:
# TODO 6: la primera fila del filtro, por posición y por etiqueta.
por_posicion = ciclistas.____[0]
por_etiqueta = ciclistas.____[ciclistas.index[0]]

print("Misma fila:", por_posicion.equals(por_etiqueta))
print()
print("Y esto, en cambio, es un error:")
try:
    ciclistas.loc[0]
except KeyError as error:
    print("  KeyError ->", error)
    print("  La etiqueta 0 no existe en este filtro: esa detección no era un ciclista.")

### ✏️ TODO 7 — ⭐ El error que **no** avisa

El `KeyError` de arriba es un buen error: se ve, se corrige y a otra cosa.

El peligroso es este. Queremos agregar la velocidad en km/h a una tabla de ciclistas con el
índice reiniciado. El código no falla. Ejecútalo y cuenta los `NaN`.

In [ ]:
kmh = ciclistas["speed_mps"] * 3.6          # conserva el índice original: 76, 194, 199, ...
tabla = ciclistas.reset_index(drop=True)    # índice nuevo: 0, 1, 2, ...

tabla["speed_kmh"] = kmh                    # pandas alinea por ETIQUETA, no por posición

print(f"Filas: {len(tabla)}")
print(f"NaN en speed_kmh: {tabla['speed_kmh'].isna().sum()}")
print(f"Filas con valor : {tabla['speed_kmh'].notna().sum()}   <- y encima están MAL")
print()
print(tabla[["object_type", "speed_mps", "speed_kmh"]].head(3))

Mira la última salida con calma:

- La operación **no dio error**.
- La mayoría de las filas quedó en `NaN`.
- **Unas pocas sí tienen valor, y ese valor está equivocado**: le corresponde a otra detección,
  la que originalmente llevaba esa etiqueta.

Un resultado parcialmente lleno es más peligroso que uno vacío: parece que funcionó.

Ahora arréglalo de las dos formas posibles.

*Pista: o bien despojas a la Series de su índice (`.to_numpy()` o `.values`), o bien reinicias el
índice de la Series igual que el del DataFrame.*

In [ ]:
# TODO 7: arregla la asignación de las dos formas.
# Forma A — quitarle el índice a la Series.
tabla_a = ciclistas.reset_index(drop=True)
tabla_a["speed_kmh"] = kmh.____()

# Forma B — reiniciar el índice de la Series igual que el del DataFrame.
tabla_b = ciclistas.reset_index(drop=True)
tabla_b["speed_kmh"] = kmh.____(drop=True)

print(f"Forma A -> NaN: {tabla_a['speed_kmh'].isna().sum()}")
print(f"Forma B -> NaN: {tabla_b['speed_kmh'].isna().sum()}")
print()
print(tabla_a[["object_type", "speed_mps", "speed_kmh"]].head(3))

In [ ]:
# Autochequeo
assert tabla["speed_kmh"].isna().sum() > 0, (
    "revisa: la versión rota debería tener NaN; ¿reiniciaste el índice del DataFrame?"
)
assert tabla_a["speed_kmh"].isna().sum() == ciclistas["speed_mps"].isna().sum(), (
    "revisa: los únicos NaN que deberían quedar son los que ya venían en speed_mps"
)
np.testing.assert_allclose(
    tabla_a["speed_kmh"].dropna().to_numpy(),
    tabla_b["speed_kmh"].dropna().to_numpy(),
    rtol=1e-9,
    err_msg="las dos formas de arreglarlo deben dar lo mismo",
)
np.testing.assert_allclose(
    tabla_a["speed_kmh"].dropna().to_numpy(),
    (tabla_a["speed_mps"].dropna() * 3.6).to_numpy(),
    rtol=1e-9,
    err_msg="cada fila debe llevar SU propia velocidad convertida",
)
print("✅ Arreglado por las dos vías, y cada fila lleva su propio valor.")

### ✏️ TODO 8 — La otra cara: posiciones de columna

`.iloc` también muerde. `df.iloc[:, 10]` devuelve la columna que esté en la **posición 10**, sea
la que sea. Si alguien reordena las columnas aguas arriba, tu código sigue funcionando y
devuelve otra cosa.

Compruébalo: mira qué columna es la 10, reordena las columnas y vuelve a mirar.

In [ ]:
# TODO 8: ¿qué pasa con .iloc cuando cambia el orden de las columnas?
print("Columna en posición 10 (original) :", df.columns[10])

barajado = df[sorted(df.columns)]
print("Columna en posición 10 (reordenado):", barajado.columns[____])
print()
print("¿df.iloc[:, 10] devuelve lo mismo en ambos?:",
      df.iloc[:, 10].equals(barajado.____[:, 10]))
print("¿df.loc[:, 'speed_mps'] devuelve lo mismo?:",
      df.loc[:, "speed_mps"].equals(barajado.____[:, "speed_mps"]))

In [ ]:
# Autochequeo
assert not df.iloc[:, 10].equals(barajado.iloc[:, 10]), (
    "revisa: al reordenar, la posición 10 debería apuntar a otra columna"
)
assert df.loc[:, "speed_mps"].equals(barajado.loc[:, "speed_mps"]), (
    "revisa: el nombre de la columna no cambia al reordenar"
)
print("✅ La posición depende del orden; la etiqueta, no.")
print("   Regla práctica: para columnas, usa SIEMPRE el nombre.")

**✍️ Tu respuesta al bloque 4:**

*(doble clic aquí y escribe)*

En una frase: ¿cuándo usarías `.iloc` y cuándo `.loc`? Da un ejemplo de cada uno.

---
# Bloque 5 · Manipulación avanzada

Tres operaciones cubren la mayor parte del trabajo diario:

| Operación | Qué hace | Pregunta que responde |
|---|---|---|
| `groupby` | Divide, aplica y combina | ¿Cuánto es X para cada categoría? |
| `merge` | Une dos tablas por una llave | ¿Cómo le pego el contexto a cada detección? |
| `pivot_table` | Cruza dos variables en una matriz | ¿Cómo se comporta X según A y B a la vez? |

### ✏️ TODO 9 — `groupby` con varias métricas

Calcula, por tipo de objeto: cantidad de detecciones, velocidad promedio, largo mediano y puntos
LiDAR medianos. Ordena de más a menos frecuente.

*Pista: `.agg(nombre=("columna", "funcion"), ...)`.*

In [ ]:
# TODO 9: resumen por tipo de objeto con cuatro métricas.
resumen_por_tipo = (
    df.groupby("____")
    .agg(
        n=("object_type", "____"),
        velocidad_media=("speed_mps", "____"),
        largo_mediano=("box_length", "____"),
        puntos_medianos=("num_lidar_points", "____"),
    )
    .sort_values("n", ascending=False)
    .round(2)
)
resumen_por_tipo

### ✏️ TODO 10 — `merge`: pegar el contexto

Cada segmento tiene un contexto (clima, momento del día). Constrúyelo como tabla aparte y únelo
al dataset por `segment_id`.

*Pista: `pd.merge(izquierda, derecha, on="llave", how="left")`. Comprueba que no se pierdan ni se
dupliquen filas.*

In [ ]:
# TODO 10: pega a cada detección el contexto de su segmento.
contexto = (
    df.groupby("____")
    .agg(n_detecciones=("segment_id", "size"),
         puntos_medianos_segmento=("num_lidar_points", "median"))
    .reset_index()
)

unido = pd.merge(df, contexto, on="____", how="____")

print(f"Antes del merge : {len(df):,} filas")
print(f"Después         : {len(unido):,} filas")
print(f"Columnas nuevas : {[c for c in unido.columns if c not in df.columns]}")
unido.head(3)

In [ ]:
# Autochequeo
assert len(unido) == len(df), (
    "revisa: un merge que cambia el número de filas indica llaves duplicadas en la tabla derecha"
)
assert unido["n_detecciones"].notna().all(), "revisa: quedaron detecciones sin contexto"
print(f"✅ {len(unido):,} filas, ninguna perdida ni duplicada.")

### ✏️ TODO 11 — `pivot_table`: cruzar dos variables

¿Los puntos LiDAR medianos dependen del clima, del momento del día, o de la combinación?
Construye la tabla cruzada.

*Pista: `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)`.*

In [ ]:
# TODO 11: puntos LiDAR medianos por momento del día y dificultad.
cruce = df.pivot_table(
    index="____",
    columns="____",
    values="num_lidar_points",
    aggfunc="____",
)
cruce

**✍️ Tu respuesta al TODO 11:**

*(doble clic aquí y escribe)*

¿Qué te dice esa tabla sobre la relación entre dificultad de detección y cantidad de puntos?
¿Es una relación que esperabas?

---
# Bloque 6 · Carga y almacenamiento eficiente

| Formato | Lectura | Escritura | Parámetro que más importa |
|---|---|---|---|
| CSV | `pd.read_csv()` | `df.to_csv()` | `index=False`, `sep`, `dtype` |
| Excel | `pd.read_excel()` | `df.to_excel()` | `sheet_name`, `index=False` |
| JSON | `pd.read_json()` | `df.to_json()` | `orient` |
| Parquet | `pd.read_parquet()` | `df.to_parquet()` | `compression` |

`index=False` merece una nota: sin él, `to_csv` escribe el índice como una columna sin nombre.
Al releer, aparece como `Unnamed: 0`. Es el origen del 90 % de las columnas basura que se ven en
los datasets de internet.

### ✏️ TODO 12 — Medir los cuatro formatos

Exporta una muestra del dataset en los cuatro formatos y compara peso y tiempos.

Usamos una **muestra de 5.000 filas** por una razón práctica: escribir 40.680 filas con
`openpyxl` tarda del orden de un minuto, y no vamos a gastar la clase mirando una barra de
progreso. (Además, Excel admite como máximo 1.048.576 filas: no es un formato para volumen.)

*Pista: `formatos.medir_formatos(df, carpeta)` hace las cuatro escrituras y las cuatro lecturas.*

In [ ]:
# TODO 12: mide los cuatro formatos sobre una muestra de 5.000 filas.
import pyarrow  # noqa: F401  (calienta el import para que no contamine la primera medición)

muestra = df_optimizado.____(5_000)
comparativa = formatos.medir_formatos(____, SALIDAS)
comparativa

### ✏️ TODO 13 — Lo que el CSV pierde por el camino

Mira la columna `conserva_dtypes` de la tabla anterior. Solo un formato dice `True`.

Averigua exactamente **qué** se perdió al pasar por CSV.

*Pista: `formatos.columnas_con_dtype_cambiado(original, releido)`.*

In [ ]:
# TODO 13: ¿qué tipos se perdieron al pasar por CSV?
muestra.to_csv(SALIDAS / "ida_y_vuelta.csv", index=____)
releida = pd.read_csv(SALIDAS / "ida_y_vuelta.csv")

perdidas = formatos.____(muestra, releida)
print(f"Columnas que cambiaron de tipo: {len(perdidas)} de {muestra.shape[1]}\n")
print(perdidas.to_string(index=False))
print()
print(f"Memoria antes de guardar : {muestra.memory_usage(deep=True).sum()/1024**2:.2f} MB")
print(f"Memoria al releer el CSV : {releida.memory_usage(deep=True).sum()/1024**2:.2f} MB")

In [ ]:
# Autochequeo
assert len(perdidas) > 0, "revisa: el CSV debería haber perdido tipos"
assert comparativa.set_index("formato").loc["parquet", "conserva_dtypes"], (
    "revisa: Parquet sí guarda el esquema"
)
assert not comparativa.set_index("formato").loc["csv", "conserva_dtypes"], (
    "revisa: el CSV no guarda tipos, solo texto"
)
print(f"✅ Todo el trabajo del TODO 4 ({len(perdidas)} columnas optimizadas) se perdió al guardar en CSV.")
print("   Parquet lo conservó.")

### ✏️ TODO 14 — El `index=False` que todos olvidan

Guarda la muestra **con** y **sin** `index=False` y compara las columnas al releer.

In [ ]:
# TODO 14: el efecto de olvidar index=False.
muestra.to_csv(SALIDAS / "con_indice.csv")                # sin index=False
muestra.to_csv(SALIDAS / "sin_indice.csv", index=____)

con = pd.read_csv(SALIDAS / "con_indice.csv")
sin = pd.read_csv(SALIDAS / "sin_indice.csv")

print("Columnas al releer 'con_indice.csv':", con.shape[1], "->", con.columns[0])
print("Columnas al releer 'sin_indice.csv':", sin.shape[1], "->", sin.columns[0])

In [ ]:
# Autochequeo
assert con.shape[1] == sin.shape[1] + 1, "revisa: sin index=False aparece una columna de más"
assert con.columns[0].startswith("Unnamed"), "revisa: la columna extra debería llamarse Unnamed: 0"
print("✅ Sin index=False, cada guardado agrega una columna basura. Guarda tres veces y tendrás tres.")

### ✏️ TODO 15 — Tu decisión

Con las cifras que acabas de medir, completa la tabla de decisiones para **tu** proyecto.

In [ ]:
# TODO 15: decide y justifica con TUS cifras.
decision = {
    "formato_para_datos_crudos": "____",
    "formato_para_datos_procesados": "____",
    "formato_para_entregar_al_negocio": "____",
    "por_que_parquet": "____",
}

for clave, valor in decision.items():
    print(f"{clave:35s}: {valor}")

In [ ]:
# Autochequeo
assert all(v and not v.startswith("____") for v in decision.values()), (
    "revisa: quedaron campos sin completar"
)
assert decision["formato_para_datos_procesados"] == "parquet", (
    "revisa: ¿qué formato conservó los tipos y pesó menos?"
)
print("✅ Decisión documentada. Cópiala a la tabla del cierre.")

---
# Cierre · Tabla de decisiones de almacenamiento

Esta es la entrega de la Actividad 1.2. Rellénala con las cifras que **tú** mediste y cópiala al
notebook del proyecto de equipo.

---

## Decisiones de estructura y almacenamiento

**Equipo:** `____`
**Dataset del proyecto:** `____`

### Lo que medí

| Medición | Mi cifra |
|---|---|
| Filas × columnas del dataset | |
| Memoria al cargarlo (`deep=True`) | |
| Memoria tras optimizar tipos | |
| Ahorro conseguido | |
| Peso en CSV | |
| Peso en Parquet | |
| Columnas que el CSV pierde de tipo | |

### Lo que decidí

| Etapa | Formato elegido | Por qué (con cifra) |
|---|---|---|
| Datos crudos | | |
| Datos procesados | | |
| Entrega al negocio | | |

### Las tres trampas que ahora conozco

1. `sys.getsizeof` sobre una lista **miente**, y `memory_usage()` sin `deep=True` también:
   `____`
2. Asignar una Series con índice distinto **no da error**, deja `NaN` y valores cruzados:
   `____`
3. Guardar en CSV **borra los tipos**: `____`

### Pregunta de cierre

Si tu dataset creciera de `____` filas a **cien millones**, ¿qué es lo primero que dejaría de
funcionar de tu código actual, y qué cambiarías?

`____`